# NB03_generate — Mô phỏng tồn kho tuần và nhật ký hao hụt (mục 3.5.2)
**Cách chạy:** mở trên Colab → Runtime → Run all.
- Đầu vào (tải theo ID từ thư mục Drive công khai của nhóm): `data/coffee_shop_sales_raw.csv`,
  `NB02_bom_final/bom.csv`, `NB02_bom_final/ingredient.csv` (của Vy, mục 3.5.1).
- Đầu ra: `staging/inventory_weekly.csv`, `staging/waste_log.csv`, `docs/planted_scenarios.md`,
  `staging/bang_3_5.md` (số liệu cho báo cáo). Ô cuối đẩy lên Drive của nhóm.
- `np.random.seed(42)`: chạy lại cho ra đúng cùng một bộ dữ liệu.

In [1]:
# ==== 0. CẤU HÌNH & THAM SỐ MÔ PHỎNG ====
import os
import numpy as np
import pandas as pd

np.random.seed(42)
rng = np.random.default_rng(42)

DRIVE_FOLDER_ID = '1Z6FJnMgtthexz27Rho5sdX5UBCIdtmYa'      # BI_K24406H_Kỳ7
FILE_IDS = {
    'coffee_shop_sales_raw.csv': '14t3nlw6cO1ju0qG7HXzWONE6XOaC9B9m',
    'bom.csv':                   '1tHTD6iWiOxTBayWvqg3z0X3ti9KysWOo',
    'ingredient.csv':            '1eW-EZo2Su3G6iZCG-5h-NDB2wHoLlYSj',
}
UPLOAD_TO_DRIVE = True

# Mỗi tham số kèm lý do (in ra Bảng tham số ở cuối notebook)
PARAMS = {
    'BASE_LOSS':        ((0.02, 0.04), 'Hao hụt nền khi pha chế/bảo quản; khoảng 2–4% theo PCCV và mục 3.2.2'),
    'RECORD_RATIO':     ((0.60, 0.70), 'Tỷ lệ phần chênh lệch được lập phiếu hủy; phần còn lại là chênh lệch chưa giải thích'),
    'SHIFT_WASTE_FACTOR': ({'Sáng': 1.0, 'Chiều': 1.0, 'Tối': 1.6},
                           'Ca Tối hủy nhiều hơn: dọn quầy cuối ngày, hủy sữa đã mở, ít nhân sự'),
    'TARGET_COVER_WEEKS': (1.5, 'Tồn đầu tuần 1 và mức tồn mục tiêu sau khi nhập = 1,5 tuần sử dụng (PCCV)'),
    'EVENT_RATE_PER_DAY': ({'milk': 1.0, 'dry': 0.4, 'pack': 0.3},
                           'Số phiếu hủy kỳ vọng mỗi ngày: sữa hay hủy lẻ; hàng khô và bao bì ít hơn'),
}
PACK_SIZE = {'g': 1000, 'ml': 1000, 'cai': 50}   # quy cách nhập: túi 1 kg, hộp 1 lít, cây 50 cái
PACK_SIZE_BY_ID = {'I024': 100}                    # nắp ly đóng 100 cái

# Kịch bản bất thường cài sẵn (PCCV bước 3)
SCENARIO_A = dict(store_id=8, ingredient_id='I008', weeks=range(10, 14), loss=(0.085, 0.095), record_ratio=(0.10, 0.20))
SCENARIO_B = dict(store_id=3, ingredient_id='I006', shift='Tối', overpour=0.06)
NOT_STOCKED = ['I020']   # nước lọc lấy từ hệ thống lọc, không nhập/tồn kho

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
WORK = '/content' if IN_COLAB else os.environ.get('NB03_BASE', '.')
DATA = os.path.join(WORK, 'data'); OUT = os.path.join(WORK, 'staging'); DOCS = os.path.join(WORK, 'docs')
for d in (DATA, OUT, DOCS):
    os.makedirs(d, exist_ok=True)
for name, fid in FILE_IDS.items():
    p = os.path.join(DATA, name)
    if not os.path.exists(p):
        import gdown
        gdown.download(id=fid, output=p, quiet=False)

Downloading...
From: https://drive.google.com/uc?id=14t3nlw6cO1ju0qG7HXzWONE6XOaC9B9m
To: /content/data/coffee_shop_sales_raw.csv
100%|██████████| 13.8M/13.8M [00:00<00:00, 55.4MB/s]
Downloading...
From: https://drive.google.com/uc?id=1tHTD6iWiOxTBayWvqg3z0X3ti9KysWOo
To: /content/data/bom.csv
100%|██████████| 15.3k/15.3k [00:00<00:00, 27.6MB/s]
Downloading...
From: https://drive.google.com/uc?id=1eW-EZo2Su3G6iZCG-5h-NDB2wHoLlYSj
To: /content/data/ingredient.csv
100%|██████████| 3.31k/3.31k [00:00<00:00, 9.52MB/s]


In [3]:
# ==== 1. ĐỌC DỮ LIỆU & CHUẨN BỊ ====
def read_csv_clean(path, **kw):
    df = pd.read_csv(path, encoding='utf-8-sig', **kw)
    df.columns = [c.strip().replace('\\_', '_') for c in df.columns]
    return df

sales = read_csv_clean(os.path.join(DATA, 'coffee_shop_sales_raw.csv'), dtype={'transaction_time': str})
bom = read_csv_clean(os.path.join(DATA, 'bom.csv'))
ing = read_csv_clean(os.path.join(DATA, 'ingredient.csv'))
for c in ('ingredient_id', 'ingredient_name', 'unit', 'quantity_basis'):
    for df in (bom, ing):
        if c in df.columns:
            df[c] = df[c].astype(str).str.strip()
assert len(sales) == 149_116, len(sales)
assert len(bom) == 178 and bom['product_id'].nunique() == 41, (len(bom), bom['product_id'].nunique())
assert (bom['quantity_basis'] == 'EP').all(), 'BOM phải ở cơ sở EP để chia yield đúng một lần'

# Quy tắc thời gian giống NB01 (mục 3.4.2)
sales['transaction_date'] = pd.to_datetime(sales['transaction_date'])
hour = pd.to_datetime(sales['transaction_time'].str.strip(), format='%H:%M:%S').dt.hour
sales['shift'] = np.select([hour <= 10, hour <= 15], ['Sáng', 'Chiều'], 'Tối')
sales['week_start'] = sales['transaction_date'] - pd.to_timedelta(sales['transaction_date'].dt.dayofweek, unit='D')
WEEK1 = sales['week_start'].min()                                    # 2022-12-26
sales['week_no'] = ((sales['week_start'] - WEEK1).dt.days // 7 + 1).astype(int)
days_obs = sales.groupby('week_no')['transaction_date'].nunique().rename('days_observed')
week_info = (sales.groupby('week_no').agg(week_start=('week_start', 'first'),
                                          first_day=('transaction_date', 'min'),
                                          last_day=('transaction_date', 'max')).join(days_obs))
stores = sales.groupby('store_id')['store_location'].first()

# Định mức quy về cơ sở mua vào: AP = EP / (yield_pct / 100) — chia yield đúng một lần
bom = bom.merge(ing[['ingredient_id', 'yield_pct']], on='ingredient_id', how='left', validate='many_to_one')
bom['quantity_ap'] = bom['quantity_ep'] / (bom['yield_pct'] / 100)

In [4]:
# ==== 2. THEORETICAL USAGE theo cửa hàng × NVL × tuần (và × ca để dùng cho kịch bản, phân bổ phiếu hủy) ====
qty = (sales.groupby(['store_id', 'week_no', 'shift', 'product_id'])['transaction_qty'].sum().reset_index())
tu_shift = (qty.merge(bom[['product_id', 'ingredient_id', 'quantity_ap']], on='product_id', how='inner')
               .assign(tu=lambda d: d['transaction_qty'] * d['quantity_ap'])
               .groupby(['store_id', 'ingredient_id', 'week_no', 'shift'])['tu'].sum().reset_index())
tu_shift = tu_shift[~tu_shift['ingredient_id'].isin(NOT_STOCKED)]
tu_wide = tu_shift.pivot_table(index=['store_id', 'ingredient_id', 'week_no'], columns='shift',
                               values='tu', fill_value=0.0).reset_index()
for s in ('Sáng', 'Chiều', 'Tối'):
    tu_wide[s] = tu_wide.get(s, 0.0)
tu_wide['theoretical_usage'] = tu_wide[['Sáng', 'Chiều', 'Tối']].sum(axis=1)

# Đối soát: TU toàn bộ từ BOM khớp với tổng (qty × quantity_ap) tính thẳng từ dòng POS
check_tu = (sales.merge(bom, on='product_id').assign(x=lambda d: d['transaction_qty'] * d['quantity_ap']))
check_tu = check_tu[~check_tu['ingredient_id'].isin(NOT_STOCKED)]['x'].sum()
assert np.isclose(check_tu, tu_wide['theoretical_usage'].sum())
used_ings = sorted(tu_wide['ingredient_id'].unique())
unused = sorted(set(ing['ingredient_id']) - set(bom['ingredient_id']))
print('NVL theo dõi tồn kho:', len(used_ings), '| NVL không có trong BOM:', unused, '| không tồn kho:', NOT_STOCKED)

NVL theo dõi tồn kho: 22 | NVL không có trong BOM: ['I019', 'I025'] | không tồn kho: ['I020']


In [6]:
# ==== 3. ACTUAL USAGE & LƯỢNG HỦY GHI NHẬN ====
df = tu_wide.merge(ing[['ingredient_id', 'ingredient_name', 'unit', 'unit_cost_usd', 'shelf_life_days']],
                   on='ingredient_id', how='left')
n = len(df)
lo, hi = PARAMS['BASE_LOSS'][0];   df['loss_rate'] = rng.uniform(lo, hi, n)
lo, hi = PARAMS['RECORD_RATIO'][0]; df['record_ratio'] = rng.uniform(lo, hi, n)
df['scenario'] = ''

mA = (df['store_id'] == SCENARIO_A['store_id']) & (df['ingredient_id'] == SCENARIO_A['ingredient_id']) \
     & df['week_no'].isin(SCENARIO_A['weeks'])
df.loc[mA, 'loss_rate'] = rng.uniform(*SCENARIO_A['loss'], mA.sum())
df.loc[mA, 'record_ratio'] = rng.uniform(*SCENARIO_A['record_ratio'], mA.sum())
df.loc[mA, 'scenario'] = 'A'

df['actual_usage'] = df['theoretical_usage'] * (1 + df['loss_rate'])
df['recorded_waste'] = df['record_ratio'] * (df['actual_usage'] - df['theoretical_usage'])

# Kịch bản B: pha vượt định mức ở ca Tối — làm tăng AU nhưng KHÔNG có phiếu hủy
mB = (df['store_id'] == SCENARIO_B['store_id']) & (df['ingredient_id'] == SCENARIO_B['ingredient_id'])
df.loc[mB, 'actual_usage'] += SCENARIO_B['overpour'] * df.loc[mB, SCENARIO_B['shift']]
df.loc[mB, 'scenario'] = 'B'

# Đơn vị "cái" phải là số nguyên
is_pc = df['unit'] == 'cai'
df.loc[is_pc, 'actual_usage'] = np.ceil(df.loc[is_pc, 'actual_usage'])
df.loc[is_pc, 'recorded_waste'] = np.floor(df.loc[is_pc, 'recorded_waste'])
df['actual_usage'] = df['actual_usage'].round(3); df['recorded_waste'] = df['recorded_waste'].round(3)
df = df.merge(week_info, left_on='week_no', right_index=True, how='left')

In [7]:
# ==== 4. NHẬT KÝ HAO HỤT (waste_log): tách lượng hủy tuần thành từng sự kiện ====
SHIFT_HOURS = {'Sáng': (6, 10), 'Chiều': (11, 15), 'Tối': (16, 20)}
def group_of(iid, unit):
    return 'milk' if iid == 'I008' else ('pack' if unit == 'cai' else 'dry')
REASONS = {
    'milk': (['Hết hạn', 'Hư hỏng/chua', 'Pha lỗi', 'Đổ vỡ/rơi vãi'], [0.35, 0.25, 0.25, 0.15]),
    'dry':  (['Pha lỗi', 'Rơi vãi', 'Hư hỏng do ẩm'], [0.50, 0.35, 0.15]),
    'pack': (['Hỏng/rách', 'Rơi bẩn'], [0.60, 0.40]),
}
events = []
for r in df.itertuples(index=False):
    W = r.recorded_waste
    if W <= 0:
        continue
    g = group_of(r.ingredient_id, r.unit)
    k = max(1, rng.poisson(PARAMS['EVENT_RATE_PER_DAY'][0][g] * r.days_observed))
    if r.unit == 'cai':
        k = int(min(k, W))
    parts = rng.dirichlet(np.ones(k)) * W
    if r.unit == 'cai':
        parts = np.floor(parts); parts[-1] += W - parts.sum()
    else:
        parts = np.round(parts, 3); parts[-1] = round(W - parts[:-1].sum(), 3)
    sw = np.array([r.Sáng, r.Chiều, r.Tối]) * np.array([PARAMS['SHIFT_WASTE_FACTOR'][0][s] for s in ('Sáng', 'Chiều', 'Tối')])
    sw = sw / sw.sum()
    days = pd.date_range(r.first_day, r.last_day)
    reasons, probs = REASONS[g]
    for q in parts:
        if q <= 0:
            continue
        sh = rng.choice(['Sáng', 'Chiều', 'Tối'], p=sw)
        h0, h1 = SHIFT_HOURS[sh]
        ts = rng.choice(days) + pd.Timedelta(hours=int(rng.integers(h0, h1 + 1)), minutes=int(rng.integers(0, 60)))
        events.append((r.store_id, r.ingredient_id, r.week_no, ts, sh, rng.choice(reasons, p=probs), q))

waste = pd.DataFrame(events, columns=['store_id', 'ingredient_id', 'week_no', 'waste_datetime', 'shift', 'reason', 'waste_qty'])
waste = waste.merge(ing[['ingredient_id', 'ingredient_name', 'unit', 'unit_cost_usd']], on='ingredient_id', how='left')
waste['store_location'] = waste['store_id'].map(stores)
waste['waste_date'] = pd.to_datetime(waste['waste_datetime']).dt.date
waste['week_start'] = waste['week_no'].map(week_info['week_start']).dt.date
waste['waste_value_usd'] = (waste['waste_qty'] * waste['unit_cost_usd']).round(4)
waste = waste.sort_values(['waste_datetime', 'store_id', 'ingredient_id']).reset_index(drop=True)
waste.insert(0, 'waste_id', [f'W{i:06d}' for i in range(1, len(waste) + 1)])
print('Số phiếu hủy:', len(waste))

Số phiếu hủy: 5042


In [8]:
# ==== 5. TỒN KHO TUẦN (inventory_weekly): tồn đầu + nhập − AU = tồn cuối ≥ 0 ====
cover = PARAMS['TARGET_COVER_WEEKS'][0]
def pack_of(iid, unit):
    return PACK_SIZE_BY_ID.get(iid, PACK_SIZE[unit])

rows, topups = [], 0
for (sid, iid), g in df.sort_values('week_no').groupby(['store_id', 'ingredient_id']):
    g = g.set_index('week_no').reindex(week_info.index)          # tuần không phát sinh → TU = AU = 0
    unit = ing.set_index('ingredient_id').at[iid, 'unit']
    pack = pack_of(iid, unit)
    full = g[week_info['days_observed'] == 7]
    daily_rate0 = full['actual_usage'].head(4).sum() / (7 * min(4, len(full)))   # mức dùng bình quân 4 tuần đủ đầu tiên
    opening = np.ceil(cover * 7 * daily_rate0 / pack) * pack                    # tồn đầu tuần 1 = 1,5 tuần sử dụng
    prev_rate = daily_rate0
    for w, r in g.iterrows():
        au = 0.0 if pd.isna(r['actual_usage']) else r['actual_usage']
        d = week_info.at[w, 'days_observed']
        target = cover * prev_rate * 7          # mức tồn mục tiêu sau khi nhập = 1,5 tuần, theo mức dùng/ngày của tuần trước
        receipt = 0.0 if w == 1 else max(0.0, np.ceil((target - opening) / pack) * pack)
        if opening + receipt < au:      # nhu cầu tuần vượt mức tồn mục tiêu → nhập bổ sung giữa tuần (gộp vào receipt, đếm để báo cáo)
            receipt += np.ceil((au - opening - receipt) / pack) * pack; topups += 1
        closing = round(opening + receipt - au, 3)
        rows.append((sid, iid, w, opening, receipt, au, closing))
        opening = closing
        if au > 0:
            prev_rate = au / d
inv = pd.DataFrame(rows, columns=['store_id', 'ingredient_id', 'week_no', 'opening_qty', 'receipt_qty', 'actual_usage', 'closing_qty'])
inv = inv.merge(df[['store_id', 'ingredient_id', 'week_no', 'theoretical_usage', 'recorded_waste', 'scenario']],
                on=['store_id', 'ingredient_id', 'week_no'], how='left')
inv[['theoretical_usage', 'recorded_waste']] = inv[['theoretical_usage', 'recorded_waste']].fillna(0.0)
inv['scenario'] = inv['scenario'].fillna('')
inv = inv.merge(ing[['ingredient_id', 'ingredient_name', 'unit', 'unit_cost_usd']], on='ingredient_id', how='left')
inv['store_location'] = inv['store_id'].map(stores)
inv['week_start'] = inv['week_no'].map(week_info['week_start']).dt.date
inv['days_observed'] = inv['week_no'].map(week_info['days_observed'])
for c in ('opening', 'receipt', 'closing'):
    inv[f'{c}_value_usd'] = (inv[f'{c}_qty'] * inv['unit_cost_usd']).round(4)
inv['actual_cogs_usd'] = (inv['actual_usage'] * inv['unit_cost_usd']).round(4)
print('Số dòng tồn kho:', len(inv), '| số lần nhập bổ sung giữa tuần:', topups)

Số dòng tồn kho: 1755 | số lần nhập bổ sung giữa tuần: 12


In [9]:
# ==== 6. KIỂM TRA ====
inv['variance'] = inv['actual_usage'] - inv['theoretical_usage']
inv['unexplained'] = inv['variance'] - inv['recorded_waste']
inv['variance_rate'] = np.where(inv['theoretical_usage'] > 0, inv['variance'] / inv['theoretical_usage'], np.nan)
inv['waste_rate'] = np.where(inv['actual_usage'] > 0, inv['recorded_waste'] / inv['actual_usage'], np.nan)

wsum = waste.groupby(['store_id', 'ingredient_id', 'week_no'])['waste_qty'].sum()
merged = inv.set_index(['store_id', 'ingredient_id', 'week_no'])['recorded_waste'].to_frame().join(wsum).fillna(0)
normal = inv[inv['scenario'] == '']
checks = pd.DataFrame([
    ('Cân bằng tồn kho lệch (tồn đầu + nhập − AU − tồn cuối ≠ 0)',
     int((inv['opening_qty'] + inv['receipt_qty'] - inv['actual_usage'] - inv['closing_qty']).abs().gt(1e-6).sum()), 0),
    ('Dòng có tồn kho âm', int((inv[['opening_qty', 'closing_qty']] < 0).any(axis=1).sum()), 0),
    ('Tổng phiếu hủy ≠ lượng hủy tuần', int((merged['recorded_waste'] - merged['waste_qty']).abs().gt(1e-6).sum()), 0),
    ('Dòng có chênh lệch chưa giải thích âm', int((inv['unexplained'] < -1e-6).sum()), 0),
    ('Phiếu hủy nằm ngoài kỳ 01/01–30/06/2023',
     int(((pd.to_datetime(waste['waste_date']) < '2023-01-01') | (pd.to_datetime(waste['waste_date']) > '2023-06-30')).sum()), 0),
], columns=['Phép kiểm', 'Kết quả', 'Kỳ vọng'])
checks['Đạt'] = checks['Kết quả'] == checks['Kỳ vọng']
print(checks.to_string(index=False))
assert checks['Đạt'].all(), 'Có phép kiểm không đạt — không ghi file'

# Kiểm chứng: quy tắc cảnh báo của 3.2.2 có bắt đúng kịch bản cài sẵn không (chỉ xét dòng có TU > 0)
v = inv[inv['theoretical_usage'] > 0]
pc_ok = v['unit'] != 'cai'                       # bao bì làm tròn số nguyên → tỷ lệ kém ổn định ở tuần biên
alert = v[(v['variance_rate'] >= 0.05) & (v['unexplained'] > 0)]
print('\nDòng vượt ngưỡng cảnh báo 5%:')
print(alert.groupby(['scenario', 'store_location', 'ingredient_name']).agg(
    so_tuan=('week_no', 'nunique'), vr_min=('variance_rate', 'min'), vr_max=('variance_rate', 'max')))

                                                 Phép kiểm  Kết quả  Kỳ vọng  Đạt
Cân bằng tồn kho lệch (tồn đầu + nhập − AU − tồn cuối ≠ 0)        0        0 True
                                        Dòng có tồn kho âm        0        0 True
                           Tổng phiếu hủy ≠ lượng hủy tuần        0        0 True
                     Dòng có chênh lệch chưa giải thích âm        0        0 True
                   Phiếu hủy nằm ngoài kỳ 01/01–30/06/2023        0        0 True

Dòng vượt ngưỡng cảnh báo 5%:
                                         so_tuan    vr_min    vr_max
scenario store_location ingredient_name                             
A        Hell's Kitchen Fresh Milk             4  0.085010  0.093997
B        Astoria        Espresso Coffee        6  0.051973  0.062097


In [10]:
# ==== 7. GHI FILE ====
INV_COLS = ['store_id', 'store_location', 'ingredient_id', 'ingredient_name', 'unit', 'week_no', 'week_start',
            'days_observed', 'opening_qty', 'receipt_qty', 'actual_usage', 'closing_qty', 'theoretical_usage',
            'recorded_waste', 'unit_cost_usd', 'opening_value_usd', 'receipt_value_usd', 'closing_value_usd',
            'actual_cogs_usd']
WASTE_COLS = ['waste_id', 'store_id', 'store_location', 'ingredient_id', 'ingredient_name', 'unit', 'waste_datetime',
              'waste_date', 'week_no', 'week_start', 'shift', 'reason', 'waste_qty', 'unit_cost_usd', 'waste_value_usd']
out_inv = inv[INV_COLS].sort_values(['store_id', 'ingredient_id', 'week_no'])
out_inv[['theoretical_usage']] = out_inv[['theoretical_usage']].round(3)
out_inv.to_csv(os.path.join(OUT, 'inventory_weekly.csv'), index=False, encoding='utf-8')
out_w = waste[WASTE_COLS].copy()
out_w['waste_datetime'] = pd.to_datetime(out_w['waste_datetime']).dt.strftime('%Y-%m-%d %H:%M:%S')
out_w.to_csv(os.path.join(OUT, 'waste_log.csv'), index=False, encoding='utf-8')
print('Đã ghi', OUT)

Đã ghi /content/staging


In [11]:
# ==== 8. TÀI LIỆU KỊCH BẢN & BẢNG CHO BÁO CÁO ====
def wk(w):
    s = week_info.at[w, 'week_start']; return f"tuần {w} ({s:%d/%m}–{s + pd.Timedelta(days=6):%d/%m/%Y})"
def vn(x, d=2):
    return f'{x:,.{d}f}'.replace(',', '#').replace('.', ',').replace('#', '.')

a = inv[inv['scenario'] == 'A']; b = inv[inv['scenario'] == 'B']
b_base = normal[normal['ingredient_id'] == SCENARIO_B['ingredient_id']]
eve_share = (df.loc[mB, 'Tối'].sum() / df.loc[mB, 'theoretical_usage'].sum())
md = f"""# Kịch bản bất thường cài sẵn (NB03_generate, seed 42)

Hệ tuần: tuần 1 bắt đầu Thứ Hai {WEEK1:%d/%m/%Y}; tuần 1 chỉ có 01/01/2023, tuần {week_info.index.max()} có {week_info['days_observed'].iloc[-1]} ngày.

## (a) Hell's Kitchen – Sữa tươi (I008) – {wk(10)} đến {wk(13)}
- Cài đặt: hao hụt ngẫu nhiên {vn(SCENARIO_A['loss'][0]*100,1)}–{vn(SCENARIO_A['loss'][1]*100,1)}% (≈ 9%) thay cho 2–4%; phiếu hủy chỉ ghi nhận {SCENARIO_A['record_ratio'][0]:.0%}–{SCENARIO_A['record_ratio'][1]:.0%} phần chênh lệch (bình thường 60–70%).
- Kết quả: Variance Rate {vn(a['variance_rate'].min()*100)}–{vn(a['variance_rate'].max()*100)}%; Waste Rate {vn(a['waste_rate'].min()*100)}–{vn(a['waste_rate'].max()*100)}%; chênh lệch chưa giải thích dương {vn(a['unexplained'].sum()/1000)} lít ({vn((a['unexplained']*a['unit_cost_usd']).sum())} USD).
- Kỳ vọng hệ thống: 4 dòng ở mức "Ưu tiên kiểm tra" (Variance Rate ≥ 8%, U > 0) trên D3.

## (b) Astoria – Hạt espresso (I006) – ca Tối pha vượt định mức {SCENARIO_B['overpour']:.0%}, cả kỳ
- Cài đặt: AU tăng thêm {SCENARIO_B['overpour']:.0%} × TU ca Tối; không lập phiếu hủy cho phần pha vượt.
- Ca Tối chiếm {vn(eve_share*100,1)}% TU espresso của Astoria, nên Variance Rate cấp tuần chỉ tăng khoảng {vn(SCENARIO_B['overpour']*eve_share*100,1)} điểm %: Astoria {vn(b['variance_rate'].mean()*100)}% so với {vn(b_base['variance_rate'].mean()*100)}% ở hai cửa hàng còn lại.
- Kết quả trên quy tắc cảnh báo 3.2.2: {(b['variance_rate']>=0.05).sum()}/{len(b)} tuần của Astoria ở mức "Cảnh báo" (Variance Rate 5–8%), không tuần nào ≥ 8%; hai cửa hàng còn lại không có tuần nào vượt 5%. Phiếu hủy espresso ca Tối của Astoria không tăng vì phần pha vượt không được lập phiếu.
- Kỳ vọng hệ thống: kịch bản này khó phát hiện hơn (a). Nó minh họa lưu ý ở 3.2.2: mức vượt 6% ở một ca không tạo tỷ lệ tuần bằng 6%; muốn khoanh vùng ca Tối phải so Unexplained Shrinkage giữa các cửa hàng, không đọc được trực tiếp từ dữ liệu kho tuần.
"""
open(os.path.join(DOCS, 'planted_scenarios.md'), 'w', encoding='utf-8').write(md)

summ = {
    'inv_rows': len(inv), 'stores': inv['store_id'].nunique(), 'ings': inv['ingredient_id'].nunique(),
    'weeks': inv['week_no'].nunique(), 'waste_rows': len(waste),
    'waste_value': waste['waste_value_usd'].sum(), 'cogs': inv['actual_cogs_usd'].sum(),
    'vr_normal': (normal.loc[normal.theoretical_usage > 0, 'variance_rate'].min(), normal.loc[normal.theoretical_usage > 0, 'variance_rate'].max()),
    'wr_normal': (normal.loc[(normal.actual_usage > 0) & (normal.unit != 'cai'), 'waste_rate'].min(),
                  normal.loc[(normal.actual_usage > 0) & (normal.unit != 'cai'), 'waste_rate'].max()),
    'wr_value': (normal['recorded_waste'] * normal['unit_cost_usd']).sum() / (normal['actual_usage'] * normal['unit_cost_usd']).sum(),
    'shift_share': waste.groupby('shift')['waste_value_usd'].sum().div(waste['waste_value_usd'].sum()).to_dict(),
    'tu_shift_share': tu_shift.assign(val=lambda d: d['tu'] * d['ingredient_id'].map(ing.set_index('ingredient_id')['unit_cost_usd']))
                              .groupby('shift')['val'].sum().pipe(lambda s: (s / s.sum()).to_dict()),
    'reason': waste.groupby('reason')['waste_value_usd'].sum().sort_values(ascending=False).round(2).to_dict(),
    'topups': topups, 'alert_rows': len(alert), 'alert_scen': alert['scenario'].value_counts().to_dict(),
}
lines = [f'{k}: {v}' for k, v in summ.items()] + ['', checks.to_markdown(index=False), '', md]
open(os.path.join(OUT, 'bang_3_5.md'), 'w', encoding='utf-8').write('\n'.join(lines))
print('\n'.join(lines))

inv_rows: 1755
stores: 3
ings: 22
weeks: 27
waste_rows: 5042
waste_value: 1918.7174
cogs: 103562.629
vr_normal: (0.020033457943925314, 0.04838709677419355)
wr_normal: (0.011867464009841892, 0.026836228104647808)
wr_value: 0.018616158467154025
shift_share: {np.str_('Chiều'): 0.288591378803361, np.str_('Sáng'): 0.39424560386016205, np.str_('Tối'): 0.31716301733647695}
tu_shift_share: {'Chiều': 0.3125891293386305, 'Sáng': 0.46674709589498387, 'Tối': 0.2206637747663857}
reason: {np.str_('Pha lỗi'): 652.33, np.str_('Rơi vãi'): 438.97, np.str_('Hỏng/rách'): 217.88, np.str_('Hư hỏng do ẩm'): 189.72, np.str_('Rơi bẩn'): 157.79, np.str_('Hết hạn'): 116.99, np.str_('Hư hỏng/chua'): 84.17, np.str_('Đổ vỡ/rơi vãi'): 60.85}
topups: 12
alert_rows: 10
alert_scen: {'B': 6, 'A': 4}

| Phép kiểm                                                  |   Kết quả |   Kỳ vọng | Đạt   |
|:-----------------------------------------------------------|----------:|----------:|:------|
| Cân bằng tồn kho lệch (tồn đầu 

In [12]:
# ==== 9. ĐẨY KẾT QUẢ LÊN DRIVE CỦA NHÓM ====
def upload_to_drive(paths, parent_id, subfolder):
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaFileUpload
    auth.authenticate_user()
    svc = build('drive', 'v3', cache_discovery=False)
    def find(name, parent, folder=False):
        q = f"name = '{name}' and '{parent}' in parents and trashed = false"
        if folder:
            q += " and mimeType = 'application/vnd.google-apps.folder'"
        r = svc.files().list(q=q, fields='files(id)', supportsAllDrives=True, includeItemsFromAllDrives=True).execute()
        return r['files'][0]['id'] if r['files'] else None
    fid_folder = find(subfolder, parent_id, folder=True) or svc.files().create(
        body={'name': subfolder, 'parents': [parent_id], 'mimeType': 'application/vnd.google-apps.folder'},
        fields='id', supportsAllDrives=True).execute()['id']
    for p in paths:
        name = os.path.basename(p)
        media = MediaFileUpload(p, mimetype='text/csv' if name.endswith('.csv') else 'text/markdown', resumable=True)
        fid = find(name, fid_folder)
        if fid:
            svc.files().update(fileId=fid, media_body=media, supportsAllDrives=True).execute(); print('Cập nhật:', name)
        else:
            fid = svc.files().create(body={'name': name, 'parents': [fid_folder]}, media_body=media,
                                     fields='id', supportsAllDrives=True).execute()['id']; print('Tạo mới:', name)
        print('   https://drive.google.com/file/d/' + fid)

if IN_COLAB and UPLOAD_TO_DRIVE:
    upload_to_drive([os.path.join(OUT, f) for f in ('inventory_weekly.csv', 'waste_log.csv', 'bang_3_5.md')],
                    DRIVE_FOLDER_ID, 'staging')
    upload_to_drive([os.path.join(DOCS, 'planted_scenarios.md')], DRIVE_FOLDER_ID, 'docs')
else:
    print('Không đẩy lên Drive; file nằm ở', OUT, 'và', DOCS)

Tạo mới: inventory_weekly.csv
   https://drive.google.com/file/d/1Im_KEKdN-ZeSCSWYW00_i6rQfLYGeX8V
Tạo mới: waste_log.csv
   https://drive.google.com/file/d/1xoaMWuvzEKOQTkJ8XR0dXRArioYTX-dz
Tạo mới: bang_3_5.md
   https://drive.google.com/file/d/1O9Vx7PReJmSAlwvmWXL1qwfNc24nuG1l


Tạo mới: planted_scenarios.md
   https://drive.google.com/file/d/1RCF6KdFFJvD91-hgFiKXtI7k05dAk2G5
